In [2]:
!pip install pandas numpy scikit-learn

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [8]:
import pandas as pd
import numpy as np

kepler = pd.read_csv("../data/processed/kepler_clean.csv")
tess = pd.read_csv("../data/processed/tess_clean.csv")

print("Kepler shape:", kepler.shape)
print("TESS shape:", tess.shape)

Kepler shape: (9564, 114)
TESS shape: (8175, 51)


In [9]:
# Potential leakage-related features

suspicious_features = [
    'koi_score',
    'koi_fpflag_nt',
    'koi_fpflag_ss',
    'koi_fpflag_co',
    'koi_fpflag_ec',
    'koi_pdisposition',
    'koi_comment'
]

# Check which features exist
available_features = [
    col for col in suspicious_features
    if col in kepler.columns
]

print("Potential leakage-related features:")
print(available_features)

# Inspect their data types and missing values
print("\nFeature Information:")

for col in available_features:
    print(f"\n{col}")
    print("Data type:", kepler[col].dtype)
    print("Missing values:", kepler[col].isnull().sum())
    print("Unique values:", kepler[col].nunique())

    print("Sample values:")
    print(kepler[col].dropna().head().tolist())

Potential leakage-related features:
['koi_score', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec', 'koi_pdisposition', 'koi_comment']

Feature Information:

koi_score
Data type: float64
Missing values: 1510
Unique values: 650
Sample values:
[1.0, 0.969, 0.0, 0.0, 1.0]

koi_fpflag_nt
Data type: int64
Missing values: 0
Unique values: 3
Sample values:
[0, 0, 0, 0, 0]

koi_fpflag_ss
Data type: int64
Missing values: 0
Unique values: 2
Sample values:
[0, 0, 0, 1, 0]

koi_fpflag_co
Data type: int64
Missing values: 0
Unique values: 2
Sample values:
[0, 0, 0, 0, 0]

koi_fpflag_ec
Data type: int64
Missing values: 0
Unique values: 2
Sample values:
[0, 0, 0, 0, 0]

koi_pdisposition
Data type: str
Missing values: 0
Unique values: 2
Sample values:
['CANDIDATE', 'CANDIDATE', 'CANDIDATE', 'FALSE POSITIVE', 'CANDIDATE']

koi_comment
Data type: str
Missing values: 1209
Unique values: 995
Sample values:
['NO_COMMENT', 'NO_COMMENT', 'DEEP_V_SHAPED', 'MOD_ODDEVEN_DV---MOD_ODDEVEN_ALT---D

In [10]:
# Target distribution across preliminary disposition
print("Preliminary Disposition vs Final Disposition")

print(pd.crosstab(
    kepler['koi_pdisposition'],
    kepler['koi_disposition']
))

# False positive flag distributions
fp_flags = [
    'koi_fpflag_nt',
    'koi_fpflag_ss',
    'koi_fpflag_co',
    'koi_fpflag_ec'
]

for col in fp_flags:
    print(f"\n{col} distribution:")
    print(pd.crosstab(
        kepler[col],
        kepler['koi_disposition']
    ))

# Score statistics by target
print("\nKOI Score Statistics:")
print(
    kepler.groupby('koi_disposition')['koi_score']
    .describe()
)

Preliminary Disposition vs Final Disposition
koi_disposition   CANDIDATE  CONFIRMED  FALSE POSITIVE
koi_pdisposition                                      
CANDIDATE              1977       2739               1
FALSE POSITIVE            0          9            4838

koi_fpflag_nt distribution:
koi_disposition  CANDIDATE  CONFIRMED  FALSE POSITIVE
koi_fpflag_nt                                        
0                     1976       2744            3313
1                        1          3            1526
465                      0          1               0

koi_fpflag_ss distribution:
koi_disposition  CANDIDATE  CONFIRMED  FALSE POSITIVE
koi_fpflag_ss                                        
0                     1975       2734            2629
1                        2         14            2210

koi_fpflag_co distribution:
koi_disposition  CANDIDATE  CONFIRMED  FALSE POSITIVE
koi_fpflag_co                                        
0                     1977       2748            2950


In [13]:
leakage_features = [
    'koi_score',
    'koi_fpflag_nt',
    'koi_fpflag_ss',
    'koi_fpflag_co',
    'koi_fpflag_ec',
    'koi_pdisposition',
    'koi_comment'
]

In [14]:
# Investigate unusual value in koi_fpflag_nt

print("Value distribution:")
print(kepler['koi_fpflag_nt'].value_counts(dropna=False))

print("\nRecord containing value 465:")

columns_to_display = [
    col for col in [
        'kepid',
        'kepoi_name',
        'koi_disposition',
        'koi_fpflag_nt'
    ]
    if col in kepler.columns
]

print(
    kepler.loc[
        kepler['koi_fpflag_nt'] == 465,
        columns_to_display
    ]
)

Value distribution:
koi_fpflag_nt
0      8033
1      1530
465       1
Name: count, dtype: int64

Record containing value 465:
         kepid kepoi_name koi_disposition  koi_fpflag_nt
3008  10934674  K00477.01       CONFIRMED            465


In [16]:
# Create working copies
kepler_processed = kepler.copy()
tess_processed = tess.copy()

# Remove leakage-related features from Kepler
kepler_processed.drop(
    columns=leakage_features,
    errors='ignore'
)

print("Original Kepler shape:", kepler.shape)
print("Processed Kepler shape:", kepler_processed.shape)

print("\nRemaining leakage features:")
print([
    col for col in leakage_features
    if col in kepler_processed.columns
])

Original Kepler shape: (9564, 114)
Processed Kepler shape: (9564, 114)

Remaining leakage features:
['koi_score', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec', 'koi_pdisposition', 'koi_comment']


In [17]:
# Missing value analysis

missing_values = pd.DataFrame({
    'Missing Count': kepler_processed.isnull().sum(),
    'Missing Percentage': (
        kepler_processed.isnull().mean() * 100
    )
})

missing_values = missing_values[
    missing_values['Missing Count'] > 0
].sort_values(
    by='Missing Percentage',
    ascending=False
)

print("Total features with missing values:",
      len(missing_values))

print("\nMissing Value Summary:")
print(missing_values.to_string())

Total features with missing values: 96

Missing Value Summary:
                    Missing Count  Missing Percentage
kepler_name                  6816           71.267252
koi_score                    1510           15.788373
koi_bin_oedp_sig             1510           15.788373
koi_comment                  1209           12.641154
koi_max_sngle_ev             1142           11.940611
koi_num_transits             1142           11.940611
koi_max_mult_ev              1142           11.940611
koi_quarters                 1142           11.940611
koi_fwm_stat_sig             1076           11.250523
koi_fwm_prao                  830            8.678377
koi_fwm_prao_err              830            8.678377
koi_fwm_pdeco                 817            8.542451
koi_fwm_pdeco_err             817            8.542451
koi_zmag                      613            6.409452
koi_dicco_mra_err             599            6.263070
koi_dicco_mra                 599            6.263070
koi_dicco_mdec_err 

In [18]:
# Categorize features based on missing percentage

print("Features with no missing values:",
      (kepler_processed.isnull().sum() == 0).sum())

print("Features with 0-5% missing values:",
      ((missing_values['Missing Percentage'] > 0) &
       (missing_values['Missing Percentage'] <= 5)).sum())

print("Features with 5-20% missing values:",
      ((missing_values['Missing Percentage'] > 5) &
       (missing_values['Missing Percentage'] <= 20)).sum())

print("Features with more than 20% missing values:",
      (missing_values['Missing Percentage'] > 20).sum())

Features with no missing values: 18
Features with 0-5% missing values: 65
Features with 5-20% missing values: 30
Features with more than 20% missing values: 1


In [19]:
# Inspect identifier and catalog-reference columns

possible_id_columns = [
    col for col in kepler_processed.columns
    if any(keyword in col.lower()
           for keyword in ['kepid', 'kepoi', 'name', 'id', 'link'])
]

print("Potential identifier/reference columns:")
print(possible_id_columns)

for col in possible_id_columns:
    print(f"\n{col}")
    print("Unique values:", kepler_processed[col].nunique())
    print("Missing values:", kepler_processed[col].isnull().sum())
    print("Sample values:",
          kepler_processed[col].dropna().head().tolist())

Potential identifier/reference columns:
['rowid', 'kepid', 'kepoi_name', 'kepler_name', 'koi_tce_delivname', 'koi_datalink_dvr', 'koi_datalink_dvs']

rowid
Unique values: 9564
Missing values: 0
Sample values: [1, 2, 3, 4, 5]

kepid
Unique values: 8214
Missing values: 0
Sample values: [10797460, 10797460, 10811496, 10848459, 10854555]

kepoi_name
Unique values: 9564
Missing values: 0
Sample values: ['K00752.01', 'K00752.02', 'K00753.01', 'K00754.01', 'K00755.01']

kepler_name
Unique values: 2748
Missing values: 6816
Sample values: ['Kepler-227 b', 'Kepler-227 c', 'Kepler-664 b', 'Kepler-228 d', 'Kepler-228 c']

koi_tce_delivname
Unique values: 3
Missing values: 346
Sample values: ['q1_q17_dr25_tce', 'q1_q17_dr25_tce', 'q1_q17_dr25_tce', 'q1_q17_dr25_tce', 'q1_q17_dr25_tce']

koi_datalink_dvr
Unique values: 8063
Missing values: 346
Sample values: ['010/010797/010797460/dv/kplr010797460-20160209194854_dvr.pdf', '010/010797/010797460/dv/kplr010797460-20160209194854_dvr.pdf', '010/010811/01

In [20]:
# Preserve Kepler IDs for future group-aware splitting
kepler_ids = kepler_processed['kepid'].copy()

# Identifier and reference columns
id_columns = [
    'rowid',
    'kepid',
    'kepoi_name',
    'kepler_name',
    'koi_tce_delivname',
    'koi_datalink_dvr',
    'koi_datalink_dvs'
]

# Remove these columns
kepler_processed = kepler_processed.drop(
    columns=id_columns,
    errors='ignore'
)

print("Kepler shape after removing identifiers:",
      kepler_processed.shape)

print("\nRemaining identifier columns:")
print([
    col for col in id_columns
    if col in kepler_processed.columns
])

Kepler shape after removing identifiers: (9564, 107)

Remaining identifier columns:
[]


In [21]:
print("Current shape:", kepler_processed.shape)

print("Target column present:",
      'koi_disposition' in kepler_processed.columns)

print("\nFirst 20 remaining columns:")
print(kepler_processed.columns[:20].tolist())

print("\nAre preserved IDs available?",
      len(kepler_ids))

Current shape: (9564, 107)
Target column present: True

First 20 remaining columns:
['koi_disposition', 'koi_pdisposition', 'koi_score', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec', 'koi_comment', 'koi_period', 'koi_period_err1', 'koi_period_err2', 'koi_time0bk', 'koi_time0bk_err1', 'koi_time0bk_err2', 'koi_time0', 'koi_time0_err1', 'koi_time0_err2', 'koi_impact', 'koi_impact_err1', 'koi_impact_err2']

Are preserved IDs available? 9564


In [22]:
# Combine leakage and identifier columns
columns_to_remove = leakage_features + id_columns

# Remove unwanted columns
kepler_processed = kepler.copy().drop(
    columns=columns_to_remove,
    errors='ignore'
)

print("Original shape:", kepler.shape)
print("Processed shape:", kepler_processed.shape)

print("\nTarget present:",
      'koi_disposition' in kepler_processed.columns)

print("\nUnwanted columns remaining:")
print([
    col for col in columns_to_remove
    if col in kepler_processed.columns
])

Original shape: (9564, 114)
Processed shape: (9564, 100)

Target present: True

Unwanted columns remaining:
[]


In [23]:
# Recalculate missing values
missing_values = pd.DataFrame({
    'Missing Count': kepler_processed.isnull().sum(),
    'Missing Percentage': kepler_processed.isnull().mean() * 100
})

missing_values = missing_values[
    missing_values['Missing Count'] > 0
].sort_values(
    by='Missing Percentage',
    ascending=False
)

print("Features with missing values:",
      len(missing_values))

print("\nMissing Value Summary:")
print(missing_values.to_string())

print("\nFeatures with no missing values:",
      (kepler_processed.isnull().sum() == 0).sum())

Features with missing values: 90

Missing Value Summary:
                    Missing Count  Missing Percentage
koi_bin_oedp_sig             1510           15.788373
koi_max_mult_ev              1142           11.940611
koi_num_transits             1142           11.940611
koi_max_sngle_ev             1142           11.940611
koi_quarters                 1142           11.940611
koi_fwm_stat_sig             1076           11.250523
koi_fwm_prao_err              830            8.678377
koi_fwm_prao                  830            8.678377
koi_fwm_pdeco_err             817            8.542451
koi_fwm_pdeco                 817            8.542451
koi_zmag                      613            6.409452
koi_dicco_mdec_err            599            6.263070
koi_dicco_msky_err            599            6.263070
koi_dicco_mra                 599            6.263070
koi_dicco_mra_err             599            6.263070
koi_dicco_mdec                599            6.263070
koi_dicco_msky           

In [24]:
# Features with more than 10% missing values

high_missing = missing_values[
    missing_values['Missing Percentage'] > 10
].copy()

print("Features with more than 10% missing:")
print(high_missing)

print("\nNumber of features:", len(high_missing))

print("\nData types:")
for col in high_missing.index:
    print(
        col,
        "|",
        kepler_processed[col].dtype,
        "| Unique values:",
        kepler_processed[col].nunique()
    )

Features with more than 10% missing:
                  Missing Count  Missing Percentage
koi_bin_oedp_sig           1510           15.788373
koi_max_mult_ev            1142           11.940611
koi_num_transits           1142           11.940611
koi_max_sngle_ev           1142           11.940611
koi_quarters               1142           11.940611
koi_fwm_stat_sig           1076           11.250523

Number of features: 6

Data types:
koi_bin_oedp_sig | float64 | Unique values: 4875
koi_max_mult_ev | float64 | Unique values: 8422
koi_num_transits | float64 | Unique values: 1627
koi_max_sngle_ev | float64 | Unique values: 8421
koi_quarters | object | Unique values: 198
koi_fwm_stat_sig | float64 | Unique values: 929


In [25]:
# Investigate high-missing features

high_missing_features = [
    'koi_bin_oedp_sig',
    'koi_max_mult_ev',
    'koi_num_transits',
    'koi_max_sngle_ev',
    'koi_quarters',
    'koi_fwm_stat_sig'
]

for col in high_missing_features:

    print("\n" + "=" * 60)
    print("FEATURE:", col)
    print("=" * 60)

    print("\nSample values:")
    print(kepler_processed[col].dropna().head(10).tolist())

    print("\nMissing percentage by target:")
    print(
        kepler_processed.groupby('koi_disposition')[col]
        .apply(lambda x: round(x.isnull().mean() * 100, 2))
    )

    if pd.api.types.is_numeric_dtype(kepler_processed[col]):

        print("\nStatistics by target:")
        print(
            kepler_processed.groupby('koi_disposition')[col]
            .median()
        )


FEATURE: koi_bin_oedp_sig

Sample values:
[0.6864, 0.0023, 0.6624, 0.0, 0.309, 0.0975, 0.546, 0.4352, 0.7213, 0.1819]

Missing percentage by target:
koi_disposition
CANDIDATE         30.35
CONFIRMED          0.62
FALSE POSITIVE    18.45
Name: koi_bin_oedp_sig, dtype: float64

Statistics by target:
koi_disposition
CANDIDATE         0.5549
CONFIRMED         0.5103
FALSE POSITIVE    0.4050
Name: koi_bin_oedp_sig, dtype: float64

FEATURE: koi_max_mult_ev

Sample values:
[28.47082, 20.109507, 187.4491, 541.8951, 33.1919, 55.204865, 33.546658, 12.325094, 46.15308, 137.44981]

Missing percentage by target:
koi_disposition
CANDIDATE         23.32
CONFIRMED          0.44
FALSE POSITIVE    13.83
Name: koi_max_mult_ev, dtype: float64

Statistics by target:
koi_disposition
CANDIDATE          9.354937
CONFIRMED         23.979681
FALSE POSITIVE    24.315005
Name: koi_max_mult_ev, dtype: float64

FEATURE: koi_num_transits

Sample values:
[142.0, 25.0, 56.0, 621.0, 515.0, 95.0, 240.0, 355.0, 185.0, 6

In [26]:
# Features excluded from the conservative baseline

high_missing_remove = [
    'koi_bin_oedp_sig',
    'koi_max_mult_ev',
    'koi_num_transits',
    'koi_max_sngle_ev',
    'koi_quarters',
    'koi_fwm_stat_sig'
]

kepler_processed = kepler_processed.drop(
    columns=high_missing_remove,
    errors='ignore'
)

print("Updated Kepler shape:", kepler_processed.shape)

print("\nRemoved features still present:")
print([
    col for col in high_missing_remove
    if col in kepler_processed.columns
])

Updated Kepler shape: (9564, 94)

Removed features still present:
[]


In [27]:
# Identify measurement uncertainty columns

error_columns = [
    col for col in kepler_processed.columns
    if '_err' in col
]

print("Total uncertainty columns:", len(error_columns))

print("\nUncertainty columns:")
print(error_columns)

print("\nMissing-value percentages:")

for col in error_columns:
    print(
        f"{col}: "
        f"{kepler_processed[col].isnull().mean() * 100:.2f}%"
    )

Total uncertainty columns: 44

Uncertainty columns:
['koi_period_err1', 'koi_period_err2', 'koi_time0bk_err1', 'koi_time0bk_err2', 'koi_time0_err1', 'koi_time0_err2', 'koi_impact_err1', 'koi_impact_err2', 'koi_duration_err1', 'koi_duration_err2', 'koi_depth_err1', 'koi_depth_err2', 'koi_ror_err1', 'koi_ror_err2', 'koi_srho_err1', 'koi_srho_err2', 'koi_prad_err1', 'koi_prad_err2', 'koi_insol_err1', 'koi_insol_err2', 'koi_dor_err1', 'koi_dor_err2', 'koi_steff_err1', 'koi_steff_err2', 'koi_slogg_err1', 'koi_slogg_err2', 'koi_smet_err1', 'koi_smet_err2', 'koi_srad_err1', 'koi_srad_err2', 'koi_smass_err1', 'koi_smass_err2', 'koi_fwm_sra_err', 'koi_fwm_sdec_err', 'koi_fwm_srao_err', 'koi_fwm_sdeco_err', 'koi_fwm_prao_err', 'koi_fwm_pdeco_err', 'koi_dicco_mra_err', 'koi_dicco_mdec_err', 'koi_dicco_msky_err', 'koi_dikco_mra_err', 'koi_dikco_mdec_err', 'koi_dikco_msky_err']

Missing-value percentages:
koi_period_err1: 4.75%
koi_period_err2: 4.75%
koi_time0bk_err1: 4.75%
koi_time0bk_err2: 4.75%


In [28]:
# Remove measurement uncertainty columns

error_columns = [
    col for col in kepler_processed.columns
    if '_err' in col
]

kepler_processed = kepler_processed.drop(
    columns=error_columns,
    errors='ignore'
)

print("Updated Kepler shape:", kepler_processed.shape)

print("\nRemaining uncertainty columns:")
print([
    col for col in kepler_processed.columns
    if '_err' in col
])

Updated Kepler shape: (9564, 50)

Remaining uncertainty columns:
[]


In [29]:
print("Remaining columns:")
for i, col in enumerate(kepler_processed.columns, 1):
    print(i, col, "-", kepler_processed[col].dtype)

Remaining columns:
1 koi_disposition - str
2 koi_period - float64
3 koi_time0bk - float64
4 koi_time0 - float64
5 koi_impact - float64
6 koi_duration - float64
7 koi_depth - float64
8 koi_ror - float64
9 koi_srho - float64
10 koi_fittype - str
11 koi_prad - float64
12 koi_sma - float64
13 koi_incl - float64
14 koi_teq - float64
15 koi_insol - float64
16 koi_dor - float64
17 koi_ldm_coeff2 - float64
18 koi_ldm_coeff1 - float64
19 koi_parm_prov - str
20 koi_model_snr - float64
21 koi_count - int64
22 koi_tce_plnt_num - float64
23 koi_steff - float64
24 koi_slogg - float64
25 koi_smet - float64
26 koi_srad - float64
27 koi_smass - float64
28 koi_sparprov - str
29 ra - float64
30 dec - float64
31 koi_kepmag - float64
32 koi_gmag - float64
33 koi_rmag - float64
34 koi_imag - float64
35 koi_zmag - float64
36 koi_jmag - float64
37 koi_hmag - float64
38 koi_kmag - float64
39 koi_fwm_sra - float64
40 koi_fwm_sdec - float64
41 koi_fwm_srao - float64
42 koi_fwm_sdeco - float64
43 koi_fwm_prao - f

In [30]:
categorical_columns = kepler_processed.select_dtypes(
    include=['object']
).columns

for col in categorical_columns:

    print(f"\nColumn: {col}")
    print("Unique values:", kepler_processed[col].nunique())

    print("\nValue counts:")
    print(kepler_processed[col].value_counts(dropna=False).head(15))


Column: koi_disposition
Unique values: 3

Value counts:
koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
CANDIDATE         1977
Name: count, dtype: int64

Column: koi_fittype
Unique values: 4

Value counts:
koi_fittype
LS+MCMC    7897
MCMC       1206
none        369
LS           92
Name: count, dtype: int64

Column: koi_parm_prov
Unique values: 3

Value counts:
koi_parm_prov
q1_q17_dr25_koi    8054
q1_q16_koi         1142
q1_q17_dr24_koi     368
Name: count, dtype: int64

Column: koi_sparprov
Unique values: 4

Value counts:
koi_sparprov
q1_q17_dr25_stellar    7907
stellar_q1_q16          956
NaN                     363
stellar_q1_q17          233
Solar                   105
Name: count, dtype: int64


C:\Users\subas\AppData\Local\Temp\ipykernel_32020\1009576973.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = kepler_processed.select_dtypes(


In [31]:
investigate_features = [
    'koi_model_snr',
    'koi_count',
    'koi_tce_plnt_num',
    'koi_srho'
]

for col in investigate_features:

    print(f"\n{'='*50}")
    print("Feature:", col)

    print("\nOverall statistics:")
    print(kepler_processed[col].describe())

    print("\nStatistics by disposition:")
    print(
        kepler_processed.groupby('koi_disposition')[col]
        .median()
    )


Feature: koi_model_snr

Overall statistics:
count    9201.000000
mean      259.895001
std       795.806615
min         0.000000
25%        12.000000
50%        23.000000
75%        78.000000
max      9054.700000
Name: koi_model_snr, dtype: float64

Statistics by disposition:
koi_disposition
CANDIDATE         11.00
CONFIRMED         28.60
FALSE POSITIVE    34.25
Name: koi_model_snr, dtype: float64

Feature: koi_count

Overall statistics:
count    9564.000000
mean        1.406315
std         0.873289
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         7.000000
Name: koi_count, dtype: float64

Statistics by disposition:
koi_disposition
CANDIDATE         1.0
CONFIRMED         2.0
FALSE POSITIVE    1.0
Name: koi_count, dtype: float64

Feature: koi_tce_plnt_num

Overall statistics:
count    9218.000000
mean        1.243654
std         0.664573
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         8.000000


In [32]:
centroid_features = [
    'koi_fwm_sra',
    'koi_fwm_sdec',
    'koi_fwm_srao',
    'koi_fwm_sdeco',
    'koi_fwm_prao',
    'koi_fwm_pdeco',
    'koi_dicco_mra',
    'koi_dicco_mdec',
    'koi_dicco_msky',
    'koi_dikco_mra',
    'koi_dikco_mdec',
    'koi_dikco_msky'
]

for col in centroid_features:

    print(f"\n{'='*45}")
    print("Feature:", col)

    print("Missing:", kepler_processed[col].isnull().sum())

    print(
        kepler_processed.groupby('koi_disposition')[col]
        .median()
    )


Feature: koi_fwm_sra
Missing: 506
koi_disposition
CANDIDATE         19.462521
CONFIRMED         19.418734
FALSE POSITIVE    19.538844
Name: koi_fwm_sra, dtype: float64

Feature: koi_fwm_sdec
Missing: 506
koi_disposition
CANDIDATE         44.082740
CONFIRMED         44.259565
FALSE POSITIVE    43.013732
Name: koi_fwm_sdec, dtype: float64

Feature: koi_fwm_srao
Missing: 455
koi_disposition
CANDIDATE        -0.02000
CONFIRMED         0.00000
FALSE POSITIVE   -0.00275
Name: koi_fwm_srao, dtype: float64

Feature: koi_fwm_sdeco
Missing: 455
koi_disposition
CANDIDATE        -0.09900
CONFIRMED        -0.01000
FALSE POSITIVE   -0.03839
Name: koi_fwm_sdeco, dtype: float64

Feature: koi_fwm_prao
Missing: 830
koi_disposition
CANDIDATE         0.00001
CONFIRMED         0.00000
FALSE POSITIVE    0.00000
Name: koi_fwm_prao, dtype: float64

Feature: koi_fwm_pdeco
Missing: 817
koi_disposition
CANDIDATE         0.000010
CONFIRMED        -0.000002
FALSE POSITIVE    0.000000
Name: koi_fwm_pdeco, dtype: f

In [33]:
print("RA statistics:")
print(kepler_processed['ra'].describe())

print("\nDEC statistics:")
print(kepler_processed['dec'].describe())

print("\nMissing values:")
print(kepler_processed[['ra', 'dec']].isnull().sum())

print("\nDisposition-wise RA median:")
print(kepler_processed.groupby('koi_disposition')['ra'].median())

print("\nDisposition-wise DEC median:")
print(kepler_processed.groupby('koi_disposition')['dec'].median())

RA statistics:
count    9564.000000
mean      292.060163
std         4.766657
min       279.852720
25%       288.660770
50%       292.261125
75%       295.859160
max       301.720760
Name: ra, dtype: float64

DEC statistics:
count    9564.000000
mean       43.810433
std         3.601243
min        36.577381
25%        40.777173
50%        43.677504
75%        46.714611
max        52.336010
Name: dec, dtype: float64

Missing values:
ra     0
dec    0
dtype: int64

Disposition-wise RA median:
koi_disposition
CANDIDATE         291.93756
CONFIRMED         291.26215
FALSE POSITIVE    293.01654
Name: ra, dtype: float64

Disposition-wise DEC median:
koi_disposition
CANDIDATE         44.082329
CONFIRMED         44.259535
FALSE POSITIVE    43.021790
Name: dec, dtype: float64


In [34]:
magnitude_features = [
    'koi_kepmag',
    'koi_gmag',
    'koi_rmag',
    'koi_imag',
    'koi_zmag',
    'koi_jmag',
    'koi_hmag',
    'koi_kmag'
]

print("Magnitude feature statistics:\n")

print(kepler_processed[magnitude_features].describe().T)

print("\nMissing values:\n")

print(kepler_processed[magnitude_features].isnull().sum())

Magnitude feature statistics:

             count       mean       std    min      25%      50%      75%  \
koi_kepmag  9563.0  14.264606  1.385448  6.966  13.4400  14.5200  15.3220   
koi_gmag    9523.0  14.830501  1.501885  7.225  13.8965  15.0640  15.9355   
koi_rmag    9555.0  14.221565  1.383713  7.101  13.3930  14.4710  15.2750   
koi_imag    9410.0  14.075138  1.292573  7.627  13.2940  14.3175  15.0630   
koi_zmag    8951.0  13.991724  1.230351  6.702  13.2760  14.2540  14.9430   
koi_jmag    9539.0  12.993311  1.291912  4.097  12.2530  13.2360  13.9680   
koi_hmag    9539.0  12.620604  1.267215  3.014  11.9145  12.8340  13.5510   
koi_kmag    9539.0  12.543410  1.268180  2.311  11.8430  12.7440  13.4845   

               max  
koi_kepmag  20.003  
koi_gmag    21.150  
koi_rmag    19.960  
koi_imag    19.900  
koi_zmag    17.403  
koi_jmag    17.372  
koi_hmag    17.615  
koi_kmag    17.038  

Missing values:

koi_kepmag      1
koi_gmag       41
koi_rmag        9
koi_imag      

In [35]:
print("Fittype vs Disposition:\n")

print(
    pd.crosstab(
        kepler_processed['koi_fittype'],
        kepler_processed['koi_disposition'],
        normalize='index'
    ).round(3) * 100
)

Fittype vs Disposition:

koi_disposition  CANDIDATE  CONFIRMED  FALSE POSITIVE
koi_fittype                                          
LS                    13.0        3.3            83.7
LS+MCMC               17.2       34.5            48.3
MCMC                  41.7        1.3            57.0
none                  28.5        0.5            71.0


In [36]:
missing_summary = pd.DataFrame({
    'Missing Count': kepler_processed.isnull().sum(),
    'Missing Percentage': (
        kepler_processed.isnull().mean() * 100
    )
})

missing_summary = missing_summary[
    missing_summary['Missing Count'] > 0
].sort_values(
    by='Missing Percentage',
    ascending=False
)

print(missing_summary)

                  Missing Count  Missing Percentage
koi_fwm_prao                830            8.678377
koi_fwm_pdeco               817            8.542451
koi_zmag                    613            6.409452
koi_dicco_mra               599            6.263070
koi_dicco_mdec              599            6.263070
koi_dicco_msky              599            6.263070
koi_dikco_mra               570            5.959849
koi_dikco_mdec              570            5.959849
koi_dikco_msky              570            5.959849
koi_fwm_sra                 506            5.290673
koi_fwm_sdec                506            5.290673
koi_fwm_srao                455            4.757424
koi_fwm_sdeco               455            4.757424
koi_smet                    386            4.035968
koi_incl                    364            3.805939
koi_impact                  363            3.795483
koi_model_snr               363            3.795483
koi_teq                     363            3.795483
koi_prad    

In [37]:
features_to_check = [
    'koi_prad',
    'koi_teq',
    'koi_impact',
    'koi_depth',
    'koi_incl',
    'koi_sma',
    'koi_model_snr',
    'koi_steff',
    'koi_srad',
    'koi_smass',
    'koi_srho',
    'koi_insol'
]

for col in features_to_check:

    print(f"\n{'='*45}")
    print("Feature:", col)

    missing_by_class = (
        kepler_processed
        .groupby('koi_disposition')[col]
        .apply(lambda x: x.isnull().mean() * 100)
    )

    print(missing_by_class.round(2))


Feature: koi_prad
koi_disposition
CANDIDATE         5.26
CONFIRMED         0.07
FALSE POSITIVE    5.31
Name: koi_prad, dtype: float64

Feature: koi_teq
koi_disposition
CANDIDATE         5.26
CONFIRMED         0.07
FALSE POSITIVE    5.31
Name: koi_teq, dtype: float64

Feature: koi_impact
koi_disposition
CANDIDATE         5.26
CONFIRMED         0.07
FALSE POSITIVE    5.31
Name: koi_impact, dtype: float64

Feature: koi_depth
koi_disposition
CANDIDATE         5.26
CONFIRMED         0.07
FALSE POSITIVE    5.31
Name: koi_depth, dtype: float64

Feature: koi_incl
koi_disposition
CANDIDATE         5.26
CONFIRMED         0.07
FALSE POSITIVE    5.33
Name: koi_incl, dtype: float64

Feature: koi_sma
koi_disposition
CANDIDATE         5.26
CONFIRMED         0.07
FALSE POSITIVE    5.31
Name: koi_sma, dtype: float64

Feature: koi_model_snr
koi_disposition
CANDIDATE         5.26
CONFIRMED         0.07
FALSE POSITIVE    5.31
Name: koi_model_snr, dtype: float64

Feature: koi_steff
koi_disposition
CANDIDA

In [38]:
# Features excluded from the conservative baseline

baseline_exclusions = [
    # Catalogue leakage / vetting information
    'koi_score',
    'koi_fpflag_nt',
    'koi_fpflag_ss',
    'koi_fpflag_co',
    'koi_fpflag_ec',
    'koi_pdisposition',
    'koi_comment',

    # Identifiers and catalogue references
    'rowid',
    'kepid',
    'kepoi_name',
    'kepler_name',
    'koi_tce_delivname',
    'koi_datalink_dvr',
    'koi_datalink_dvs',

    # Observation / vetting metadata
    'koi_bin_oedp_sig',
    'koi_max_mult_ev',
    'koi_num_transits',
    'koi_max_sngle_ev',
    'koi_quarters',
    'koi_fwm_stat_sig',

    # Catalogue provenance / fitting metadata
    'koi_parm_prov',
    'koi_sparprov',
    'koi_tce_plnt_num',
    'koi_fittype',

    # Centroid-based vetting features
    'koi_fwm_sra',
    'koi_fwm_sdec',
    'koi_fwm_srao',
    'koi_fwm_sdeco',
    'koi_fwm_prao',
    'koi_fwm_pdeco',
    'koi_dicco_mra',
    'koi_dicco_mdec',
    'koi_dicco_msky',
    'koi_dikco_mra',
    'koi_dikco_mdec',
    'koi_dikco_msky',

    # Sky coordinates
    'ra',
    'dec'
]

# Create a fresh baseline dataframe
kepler_baseline = kepler_processed.drop(
    columns=baseline_exclusions,
    errors='ignore'
).copy()

print("Original processed shape:", kepler_processed.shape)
print("Baseline shape:", kepler_baseline.shape)

print("\nRemaining columns:")
print(kepler_baseline.columns.tolist())

Original processed shape: (9564, 50)
Baseline shape: (9564, 32)

Remaining columns:
['koi_disposition', 'koi_period', 'koi_time0bk', 'koi_time0', 'koi_impact', 'koi_duration', 'koi_depth', 'koi_ror', 'koi_srho', 'koi_prad', 'koi_sma', 'koi_incl', 'koi_teq', 'koi_insol', 'koi_dor', 'koi_ldm_coeff2', 'koi_ldm_coeff1', 'koi_model_snr', 'koi_count', 'koi_steff', 'koi_slogg', 'koi_smet', 'koi_srad', 'koi_smass', 'koi_kepmag', 'koi_gmag', 'koi_rmag', 'koi_imag', 'koi_zmag', 'koi_jmag', 'koi_hmag', 'koi_kmag']


In [39]:
# Check data types
print("DATA TYPES")
print(kepler_baseline.dtypes)

# Check missing values
print("\nMISSING VALUES")

missing_summary = pd.DataFrame({
    'Missing Count': kepler_baseline.isnull().sum(),
    'Missing Percentage': kepler_baseline.isnull().mean() * 100
})

missing_summary = missing_summary[
    missing_summary['Missing Count'] > 0
].sort_values(by='Missing Percentage', ascending=False)

print(missing_summary)

# Check target distribution
print("\nTARGET DISTRIBUTION")
print(kepler_baseline['koi_disposition'].value_counts())

DATA TYPES
koi_disposition        str
koi_period         float64
koi_time0bk        float64
koi_time0          float64
koi_impact         float64
koi_duration       float64
koi_depth          float64
koi_ror            float64
koi_srho           float64
koi_prad           float64
koi_sma            float64
koi_incl           float64
koi_teq            float64
koi_insol          float64
koi_dor            float64
koi_ldm_coeff2     float64
koi_ldm_coeff1     float64
koi_model_snr      float64
koi_count            int64
koi_steff          float64
koi_slogg          float64
koi_smet           float64
koi_srad           float64
koi_smass          float64
koi_kepmag         float64
koi_gmag           float64
koi_rmag           float64
koi_imag           float64
koi_zmag           float64
koi_jmag           float64
koi_hmag           float64
koi_kmag           float64
dtype: object

MISSING VALUES
                Missing Count  Missing Percentage
koi_zmag                  613            6.40

In [40]:
# Separate input features and target

X = kepler_baseline.drop(columns=['koi_disposition'])
y = kepler_baseline['koi_disposition']

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget classes:")
print(y.unique())

Feature matrix shape: (9564, 31)
Target vector shape: (9564,)

Feature columns:
['koi_period', 'koi_time0bk', 'koi_time0', 'koi_impact', 'koi_duration', 'koi_depth', 'koi_ror', 'koi_srho', 'koi_prad', 'koi_sma', 'koi_incl', 'koi_teq', 'koi_insol', 'koi_dor', 'koi_ldm_coeff2', 'koi_ldm_coeff1', 'koi_model_snr', 'koi_count', 'koi_steff', 'koi_slogg', 'koi_smet', 'koi_srad', 'koi_smass', 'koi_kepmag', 'koi_gmag', 'koi_rmag', 'koi_imag', 'koi_zmag', 'koi_jmag', 'koi_hmag', 'koi_kmag']

Target classes:
<StringArray>
['CONFIRMED', 'CANDIDATE', 'FALSE POSITIVE']
Length: 3, dtype: str


In [41]:
# Retrieve star IDs while preserving row alignment

groups = kepler.loc[X.index, 'kepid']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Groups shape:", groups.shape)

print("\nMissing star IDs:", groups.isnull().sum())
print("Unique stars:", groups.nunique())
print("Total observations:", len(groups))

# Check stars with multiple planet records
print("\nStars with multiple observations:")
print(groups.value_counts().value_counts().sort_index())

X shape: (9564, 31)
y shape: (9564,)
Groups shape: (9564,)

Missing star IDs: 0
Unique stars: 8214
Total observations: 9564

Stars with multiple observations:
count
1    7276
2     661
3     179
4      68
5      25
6       3
7       2
Name: count, dtype: int64


In [42]:
from sklearn.model_selection import StratifiedGroupKFold

# Initialize group-aware stratified splitter
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Generate train-test indices
train_idx, test_idx = next(
    sgkf.split(X, y, groups=groups)
)

# Create training and testing datasets
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

# Display shapes
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target:", y_train.shape)
print("Testing target:", y_test.shape)

# Check for overlapping stars
overlap = set(groups_train) & set(groups_test)

print("\nOverlapping stars:", len(overlap))

# Compare class distributions
print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Training features: (7650, 31)
Testing features: (1914, 31)

Training target: (7650,)
Testing target: (1914,)

Overlapping stars: 0

Training class distribution:
koi_disposition
FALSE POSITIVE    50.60
CONFIRMED         28.73
CANDIDATE         20.67
Name: proportion, dtype: float64

Testing class distribution:
koi_disposition
FALSE POSITIVE    50.57
CONFIRMED         28.74
CANDIDATE         20.69
Name: proportion, dtype: float64


In [43]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Numerical preprocessing pipeline
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Fit only on training data
X_train_processed = numeric_pipeline.fit_transform(X_train)

# Transform test data using training statistics
X_test_processed = numeric_pipeline.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

print("\nMissing values in training:",
      np.isnan(X_train_processed).sum())

print("Missing values in testing:",
      np.isnan(X_test_processed).sum())

Processed training shape: (7650, 31)
Processed testing shape: (1914, 31)

Missing values in training: 0
Missing values in testing: 0


In [44]:
# Check means and standard deviations after scaling

print("Training feature means (first 10):")
print(np.mean(X_train_processed, axis=0)[:10].round(4))

print("\nTraining feature standard deviations (first 10):")
print(np.std(X_train_processed, axis=0)[:10].round(4))

Training feature means (first 10):
[ 0.  0. -0. -0.  0.  0.  0.  0. -0.  0.]

Training feature standard deviations (first 10):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [49]:
os.makedirs("../models", exist_ok=True)

In [48]:
import os
import joblib

# Create directories first
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save processed datasets
np.save("../data/processed/X_train.npy", X_train_processed)
np.save("../data/processed/X_test.npy", X_test_processed)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

groups_train.to_csv("../data/processed/groups_train.csv", index=False)
groups_test.to_csv("../data/processed/groups_test.csv", index=False)

# Save fitted preprocessing pipeline
joblib.dump(
    numeric_pipeline,
    "../models/kepler_preprocessing_pipeline.pkl"
)

print("All preprocessing artifacts saved successfully!")

All preprocessing artifacts saved successfully!
